# 05 — GAN 3D Unnormalized
**Dự án:** Latent Manipulation of Brain MRI using Volume-Preserving GANs

**Input:** NIfTI unnormalized từ `00b_preprocessing_3d.ipynb`

**Kiến trúc:**
- **Generator**: 3D U-Net + Age Embedding inject vào bottleneck
- **Discriminator**: 3D PatchGAN + Age Conditioning

**Output:**
```
gan3d_unnormalized/
└── best_model.pth
```

## Bước 1: Cấu hình

In [3]:
import os

# ==== ĐƯỜNG DẪN ====
# Khác file 04: trỏ tới unnormalized thay vì normalized
DATA_DIR   = '/kaggle/input/datasets/minhbodoi/3000-preprocessed-3d/preprocessed_3d/unnormalized'
LABELS_CSV = '/kaggle/input/datasets/minhbodoi/3000-preprocessed-3d/preprocessed_3d/preprocessing_log.csv'
OUTPUT_DIR = '/kaggle/working/conditional_gan3d_unnormalized'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==== HYPERPARAMETERS ====
VOLUME_SIZE = 64
BATCH_SIZE  = 1
NUM_EPOCHS  = 300
PATIENCE    = 20   # dừng nếu val_SSIM không tăng >= MIN_DELTA sau 20 epoch liên tiếp
LR_G        = 2e-4
LR_D        = 2e-4
LAMBDA_L1   = 10   # giảm từ 100 → 10
LAMBDA_AGE  = 5    # tăng từ 1 → 5
LAMBDA_GP   = 10   # gradient penalty WGAN-GP
LATENT_DIM  = 256

nii_files = [f for f in os.listdir(DATA_DIR)
             if f.endswith('.nii.gz') or f.endswith('.nii')]
print(f'Data dir : {DATA_DIR}')
print(f'NII files: {len(nii_files)}')

Data dir : /kaggle/input/datasets/minhbodoi/3000-preprocessed-3d/preprocessed_3d/unnormalized
NII files: 3060


## Bước 2: Import thư viện

In [4]:
!pip install nibabel -q

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import nibabel as nib
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Device: cuda
GPU : Tesla T4
VRAM: 15.6 GB


## Bước 3: Dataset

In [5]:
def find_nii(data_dir, subject_id):
    """Tìm file .nii hoặc .nii.gz — xử lý cả 2 trường hợp."""
    for ext in ['.nii.gz', '.nii']:
        path = os.path.join(data_dir, f'{subject_id}{ext}')
        if os.path.exists(path):
            return path
    return None


class BrainMRI3DDataset(Dataset):
    def __init__(self, data_dir, labels_csv, volume_size=64):
        self.data_dir    = data_dir
        self.volume_size = volume_size

        df = pd.read_csv(labels_csv)
        df['nii_path'] = df['subject_id'].apply(lambda x: find_nii(data_dir, x))
        df = df[df['nii_path'].notna()].reset_index(drop=True)

        self.df      = df
        self.age_min = df['age'].min()
        self.age_max = df['age'].max()

        print(f'Dataset: {len(df)} subjects | tuổi [{self.age_min:.1f}, {self.age_max:.1f}]')

    def normalize_age(self, age):
        return 2 * (age - self.age_min) / (self.age_max - self.age_min) - 1

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        data = nib.load(row['nii_path']).get_fdata().astype(np.float32)
        vol  = torch.tensor(data).unsqueeze(0).unsqueeze(0)
        vol  = F.interpolate(vol, size=(self.volume_size,) * 3,
                             mode='trilinear', align_corners=False)
        vol  = vol.squeeze(0) * 2 - 1  # [-1, 1]
        age_norm = torch.tensor(self.normalize_age(row['age']), dtype=torch.float32)
        age_raw  = torch.tensor(row['age'] / 100.0,            dtype=torch.float32)
        return vol, age_norm, age_raw


dataset    = BrainMRI3DDataset(DATA_DIR, LABELS_CSV, VOLUME_SIZE)

# ── Train / Test split 80/20 ──────────────────────────────────
from torch.utils.data import random_split

full_dataset = BrainMRI3DDataset(DATA_DIR, LABELS_CSV, VOLUME_SIZE)
n_total = len(full_dataset)
n_train = int(0.8 * n_total)
n_test  = n_total - n_train

train_set, test_set = random_split(
    full_dataset, [n_train, n_test],
    generator=torch.Generator().manual_seed(42)
)

dataset         = full_dataset  # để lấy age_min/max
dataloader      = DataLoader(train_set, batch_size=BATCH_SIZE,
                             shuffle=True,  num_workers=2, pin_memory=True)
dataloader_test = DataLoader(test_set,  batch_size=1,
                             shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {n_train} subjects | Test: {n_test} subjects')


Dataset: 3060 subjects | tuổi [18.0, 88.0]
Dataset: 3060 subjects | tuổi [18.0, 88.0]
Train: 2448 subjects | Test: 612 subjects


## Bước 4: Kiến trúc Model 3D

In [6]:
class AgeEmbedding(nn.Module):
    def __init__(self, embed_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 128), nn.ReLU(),
            nn.Linear(128, embed_dim)
        )
    def forward(self, age):
        return self.net(age.unsqueeze(-1))


class UNetBlock3D(nn.Module):
    def __init__(self, in_ch, out_ch, down=True, use_bn=True, dropout=False):
        super().__init__()
        layers = []
        if down : layers.append(nn.Conv3d(in_ch, out_ch, 4, 2, 1, bias=False))
        else    : layers.append(nn.ConvTranspose3d(in_ch, out_ch, 4, 2, 1, bias=False))
        if use_bn  : layers.append(nn.BatchNorm3d(out_ch))
        if dropout : layers.append(nn.Dropout(0.5))
        layers.append(nn.LeakyReLU(0.2) if down else nn.ReLU())
        self.block = nn.Sequential(*layers)
    def forward(self, x):
        return self.block(x)


class Generator3D(nn.Module):
    """
    3D U-Net Generator với age conditioning.
    Input : volume (B, 1, 64, 64, 64) + tuổi (B,)
    Output: volume sinh ra (B, 1, 64, 64, 64)
    """
    def __init__(self, latent_dim=256):
        super().__init__()
        self.age_embed = AgeEmbedding(latent_dim)
        self.age_proj  = nn.Linear(latent_dim, 256)
        self.age_fuse  = nn.Conv3d(512, 256, 1)  # concat e4(256) + age(256) → 256
        self.e1 = UNetBlock3D(1,   32,  down=True, use_bn=False)
        self.e2 = UNetBlock3D(32,  64,  down=True)
        self.e3 = UNetBlock3D(64,  128, down=True)
        self.e4 = UNetBlock3D(128, 256, down=True, use_bn=False)
        self.d1 = UNetBlock3D(256, 128, down=False, dropout=True)
        self.d2 = UNetBlock3D(256, 64,  down=False)
        self.d3 = UNetBlock3D(128, 32,  down=False)
        self.out = nn.Sequential(nn.ConvTranspose3d(64, 1, 4, 2, 1), nn.Tanh())

    def forward(self, x, age):
        e1 = self.e1(x); e2 = self.e2(e1); e3 = self.e3(e2); e4 = self.e4(e3)
        age_feat = self.age_proj(self.age_embed(age)).view(-1, 256, 1, 1, 1).expand_as(e4)
        z  = self.age_fuse(torch.cat([e4, age_feat], dim=1))  # concat → model tự học balance
        d1 = self.d1(z)
        d2 = self.d2(torch.cat([d1, e3], dim=1))
        d3 = self.d3(torch.cat([d2, e2], dim=1))
        return self.out(torch.cat([d3, e1], dim=1))


class Discriminator3D(nn.Module):
    """
    3D PatchGAN Discriminator với age conditioning.
    Input : volume (B, 1, 64, 64, 64) + age channel → (B, 2, 64, 64, 64)
    """
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv3d(2,  32,  4, 2, 1, bias=False), nn.LeakyReLU(0.2),
            nn.Conv3d(32, 64,  4, 2, 1, bias=False), nn.BatchNorm3d(64),  nn.LeakyReLU(0.2),
            nn.Conv3d(64, 128, 4, 2, 1, bias=False), nn.BatchNorm3d(128), nn.LeakyReLU(0.2),
            nn.Conv3d(128, 1,  4, 1, 1)
        )
    def forward(self, vol, age):
        age_map = age.view(-1, 1, 1, 1, 1).expand(
            -1, 1, vol.shape[2], vol.shape[3], vol.shape[4]
        )
        return self.model(torch.cat([vol, age_map], dim=1))


G = Generator3D(LATENT_DIM).to(DEVICE)
D = Discriminator3D().to(DEVICE)
print(f'Generator3D    : {sum(p.numel() for p in G.parameters())/1e6:.1f}M params')
print(f'Discriminator3D: {sum(p.numel() for p in D.parameters())/1e6:.1f}M params')

Generator3D    : 6.4M params
Discriminator3D: 0.7M params


## Bước 5: Loss + Optimizers

In [7]:
# ── WGAN-GP ──────────────────────────────────────────────────
criterion_l1  = nn.L1Loss()
criterion_age = nn.MSELoss()

def compute_gradient_penalty_3d(D, real, fake, ages, device):
    B = real.size(0)
    alpha = torch.rand(B, 1, 1, 1, 1, device=device)
    interp = (alpha * real + (1 - alpha) * fake).requires_grad_(True)
    d_interp = D(interp, ages)
    grad = torch.autograd.grad(
        outputs=d_interp, inputs=interp,
        grad_outputs=torch.ones_like(d_interp),
        create_graph=True, retain_graph=True
    )[0]
    gp = ((grad.norm(2, dim=1) - 1) ** 2).mean()
    return gp

age_regressor = nn.Sequential(
    nn.AdaptiveAvgPool3d(4), nn.Flatten(),
    nn.Linear(64, 64), nn.ReLU(),
    nn.Linear(64, 1), nn.Sigmoid()
).to(DEVICE)

opt_G = optim.Adam(
    list(G.parameters()) + list(age_regressor.parameters()),
    lr=LR_G, betas=(0.0, 0.9)
)
opt_D = optim.Adam(D.parameters(), lr=LR_D, betas=(0.0, 0.9))

scheduler_G = optim.lr_scheduler.StepLR(opt_G, step_size=50, gamma=0.5)
scheduler_D = optim.lr_scheduler.StepLR(opt_D, step_size=50, gamma=0.5)

print('Loss + Optimizers (WGAN-GP 3D) sẵn sàng!')


Loss + Optimizers (WGAN-GP 3D) sẵn sàng!


## Bước 6: Training

In [8]:
from skimage.metrics import structural_similarity as ssim_fn
import numpy as np, subprocess, shutil, tempfile

# ── Duong dan checkpoint ──────────────────────────────────────
CKPT_PATH    = f'{OUTPUT_DIR}/last_checkpoint.pth'
BEST_PATH    = f'{OUTPUT_DIR}/best_model.pth'

# ── Kaggle Dataset info ───────────────────────────────────────
_ku = subprocess.run('kaggle config view', shell=True, capture_output=True, text=True).stdout
KAGGLE_USER  = [l.split(':')[1].strip() for l in _ku.split('\n') if 'username' in l][0]
DATASET_NAME = os.path.basename(OUTPUT_DIR).replace('_', '-')
MIN_DELTA    = 0.005  # chi tinh la cai thien that khi SSIM tang >= 0.005

def push_checkpoints_to_kaggle(msg):
    tmp = tempfile.mkdtemp()
    try:
        for fn in ['last_checkpoint.pth', 'best_model.pth']:
            if os.path.exists(f'{OUTPUT_DIR}/{fn}'):
                shutil.copy2(f'{OUTPUT_DIR}/{fn}', f'{tmp}/{fn}')
        import json as _j
        with open(f'{tmp}/dataset-metadata.json', 'w') as _f:
            _j.dump({'title': DATASET_NAME,
                     'id'   : f'{KAGGLE_USER}/{DATASET_NAME}',
                     'licenses': [{'name': 'CC0-1.0'}]}, _f)
        chk = subprocess.run(
            f'kaggle datasets list --user {KAGGLE_USER} --search {DATASET_NAME}',
            shell=True, capture_output=True, text=True)
        if DATASET_NAME in chk.stdout:
            subprocess.run(f'kaggle datasets version -p {tmp} -m "{msg}"', shell=True)
        else:
            subprocess.run(f'kaggle datasets create -p {tmp}', shell=True)
        print(f'  Cloud: {msg} -> {KAGGLE_USER}/{DATASET_NAME}')
    finally:
        shutil.rmtree(tmp)

# ── Resume neu co checkpoint ──────────────────────────────────
start_epoch    = 1
patience_count = 0
best_val_ssim  = -1.0
history = {'loss_G': [], 'loss_D': [], 'loss_L1': [], 'loss_age': [], 'val_ssim': []}

def find_checkpoint(filename):
    p = f'{OUTPUT_DIR}/{filename}'
    if os.path.exists(p): return p, 'working'
    import glob
    matches = glob.glob(f'/kaggle/input/**/{filename}', recursive=True)
    if matches: return matches[0], 'input'
    return None, None

load_path = None
for fname in ['last_checkpoint.pth', 'best_model.pth']:
    p, src_loc = find_checkpoint(fname)
    if p:
        load_path = p
        print(f'Tim thay {fname} ({src_loc}) — load de train tiep...')
        break
if not load_path:
    print('Khong co checkpoint — train tu dau')

if load_path:
    ckpt = torch.load(load_path, map_location=DEVICE)
    G.load_state_dict(ckpt['G_state'])
    D.load_state_dict(ckpt['D_state'])
    opt_G.load_state_dict(ckpt['opt_G'])
    opt_D.load_state_dict(ckpt['opt_D'])
    start_epoch    = ckpt['epoch'] + 1
    best_val_ssim  = ckpt.get('best_val_ssim', -1.0)
    patience_count = ckpt.get('patience_count', 0)
    history        = ckpt.get('history', history)
    print(f'Resume tu epoch {ckpt["epoch"]} | best_val_SSIM={best_val_ssim:.4f}')
    print(f'   Tiep tuc tu epoch {start_epoch} -> {NUM_EPOCHS}')

N_CRITIC = 5
print(f'Bat dau training {NUM_EPOCHS} epochs (WGAN-GP 3D)...\n')

for epoch in range(start_epoch, NUM_EPOCHS + 1):
    G.train(); D.train()
    epoch_loss_G = epoch_loss_D = epoch_loss_L1 = epoch_loss_age = 0
    n_batches = 0

    data_iter = iter(dataloader)
    for _ in range(max(len(dataloader) // N_CRITIC, 1)):
        for _ in range(N_CRITIC):
            try:
                real_vols, ages_norm, ages_raw = next(data_iter)
            except StopIteration:
                data_iter = iter(dataloader)
                real_vols, ages_norm, ages_raw = next(data_iter)
            real_vols = real_vols.to(DEVICE)
            ages_norm = ages_norm.to(DEVICE)
            ages_raw  = ages_raw.to(DEVICE)
            opt_D.zero_grad()
            with torch.no_grad():
                fake_vols = G(real_vols, ages_norm)
            loss_D_real = -D(real_vols, ages_norm).mean()
            loss_D_fake =  D(fake_vols.detach(), ages_norm).mean()
            gp = compute_gradient_penalty_3d(D, real_vols, fake_vols.detach(),
                                             ages_norm, DEVICE)
            loss_D = loss_D_real + loss_D_fake + LAMBDA_GP * gp
            loss_D.backward()
            opt_D.step()
            epoch_loss_D += loss_D.item()

        try:
            real_vols, ages_norm, ages_raw = next(data_iter)
        except StopIteration:
            data_iter = iter(dataloader)
            real_vols, ages_norm, ages_raw = next(data_iter)
        real_vols = real_vols.to(DEVICE)
        ages_norm = ages_norm.to(DEVICE)
        ages_raw  = ages_raw.to(DEVICE)
        opt_G.zero_grad()
        fake_vols  = G(real_vols, ages_norm)
        loss_G_adv = -D(fake_vols, ages_norm).mean()
        loss_L1    = criterion_l1(fake_vols, real_vols) * LAMBDA_L1
        pred_age   = age_regressor(fake_vols).squeeze()
        loss_age   = criterion_age(pred_age, ages_raw) * LAMBDA_AGE
        loss_G     = loss_G_adv + loss_L1 + loss_age
        loss_G.backward()
        opt_G.step()
        epoch_loss_G   += loss_G_adv.item()
        epoch_loss_L1  += loss_L1.item()
        epoch_loss_age += loss_age.item()
        n_batches += 1

    scheduler_G.step()
    scheduler_D.step()

    n = max(n_batches, 1)
    avg_loss_G   = epoch_loss_G   / n
    avg_loss_D   = epoch_loss_D   / (n * N_CRITIC)
    avg_loss_L1  = epoch_loss_L1  / n
    avg_loss_age = epoch_loss_age / n

    # ── SSIM tren test set voi shuffled age ───────────────────
    G.eval()
    ssim_scores = []
    with torch.no_grad():
        all_vols, all_ages = [], []
        for real_vols, ages_norm, _ in dataloader_test:
            all_vols.append(real_vols)
            all_ages.append(ages_norm)
        all_vols      = torch.cat(all_vols, dim=0)
        all_ages      = torch.cat(all_ages, dim=0)
        shuffled_idx  = torch.randperm(len(all_ages))
        all_ages_shuf = all_ages[shuffled_idx]
        for i in range(len(all_vols)):
            vol  = all_vols[i:i+1].to(DEVICE)
            age  = all_ages_shuf[i:i+1].to(DEVICE)
            fake = G(vol, age)
            r = (vol[0, 0].cpu().numpy() + 1) / 2
            f = (fake[0, 0].cpu().numpy() + 1) / 2
            slice_scores = [ssim_fn(r[j], f[j], data_range=1.0)
                            for j in range(r.shape[0])]
            ssim_scores.append(float(np.mean(slice_scores)))
    val_ssim = float(np.mean(ssim_scores))

    history['loss_G'].append(avg_loss_G)
    history['loss_D'].append(avg_loss_D)
    history['loss_L1'].append(avg_loss_L1)
    history['loss_age'].append(avg_loss_age)
    history['val_ssim'].append(val_ssim)

    # ── Luu last_checkpoint ───────────────────────────────────
    torch.save({
        'epoch'         : epoch,
        'G_state'       : G.state_dict(),
        'D_state'       : D.state_dict(),
        'opt_G'         : opt_G.state_dict(),
        'opt_D'         : opt_D.state_dict(),
        'history'       : history,
        'age_min'       : dataset.age_min,
        'age_max'       : dataset.age_max,
        'best_val_ssim' : best_val_ssim,
        'patience_count': patience_count,
        'test_indices'  : test_set.indices,
    }, CKPT_PATH)

    # ── Luu best model neu cai thien ──────────────────────────
    if val_ssim > best_val_ssim + MIN_DELTA:
        best_val_ssim  = val_ssim
        patience_count = 0
        torch.save({
            'epoch'         : epoch,
            'G_state'       : G.state_dict(),
            'D_state'       : D.state_dict(),
            'opt_G'         : opt_G.state_dict(),
            'opt_D'         : opt_D.state_dict(),
            'history'       : history,
            'age_min'       : dataset.age_min,
            'age_max'       : dataset.age_max,
            'best_loss_G'   : avg_loss_G,
            'best_val_ssim' : best_val_ssim,
            'val_ssim'      : val_ssim,
            'test_indices'  : test_set.indices,
        }, BEST_PATH)
        print(f'Epoch {epoch:3d}/{NUM_EPOCHS} | '
              f'loss_G={avg_loss_G:.4f} | loss_D={avg_loss_D:.4f} | '
              f'loss_L1={avg_loss_L1:.4f} | val_SSIM={val_ssim:.4f} | [BEST]')
    else:
        patience_count += 1
        print(f'Epoch {epoch:3d}/{NUM_EPOCHS} | '
              f'loss_G={avg_loss_G:.4f} | loss_D={avg_loss_D:.4f} | '
              f'loss_L1={avg_loss_L1:.4f} | val_SSIM={val_ssim:.4f} | '
              f'no improve {patience_count}/{PATIENCE}')

    # ── Push moi epoch ────────────────────────────────────────
    push_checkpoints_to_kaggle(f'{epoch}/{NUM_EPOCHS}')

    # ── Early stopping ────────────────────────────────────────
    if patience_count >= PATIENCE:
        print(f'Early stopping tai epoch {epoch}')
        break

print(f'\n=== TRAINING HOAN THANH ===')
print(f'Best val_SSIM: {best_val_ssim:.4f}')
print(f'Model luu tai: {OUTPUT_DIR}/best_model.pth')


Khong co checkpoint — train tu dau
Bat dau training 300 epochs (WGAN-GP 3D)...

Epoch   1/300 | loss_G=-3.4359 | loss_D=-24.1084 | loss_L1=1.0759 | val_SSIM=0.8913 | [BEST]
Starting upload for file best_model.pth


100%|██████████| 81.0M/81.0M [00:01<00:00, 59.6MB/s]
  0%|          | 0.00/81.0M [00:00<?, ?B/s]

Upload successful: best_model.pth (81MB)
Starting upload for file last_checkpoint.pth


100%|██████████| 81.0M/81.0M [00:00<00:00, 87.8MB/s]


Upload successful: last_checkpoint.pth (81MB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/dyio147/conditional-gan3d-unnormalized
  Cloud: 1/300 -> dyio147/conditional-gan3d-unnormalized
Epoch   2/300 | loss_G=-42.2764 | loss_D=-7.8994 | loss_L1=0.4490 | val_SSIM=0.6724 | no improve 1/20
Starting upload for file best_model.pth


...

Upload successful: last_checkpoint.pth (81MB)
Dataset version is being created. Please check progress at https://www.kaggle.com/datasets/dyio147/conditional-gan3d-unnormalized
  Cloud: 42/300 -> dyio147/conditional-gan3d-unnormalized
Epoch  43/300 | loss_G=140.9173 | loss_D=0.7907 | loss_L1=0.1605 | val_SSIM=0.9692 | no improve 20/20
Starting upload for file best_model.pth


100%|██████████| 81.0M/81.0M [00:01<00:00, 57.0MB/s]
  0%|          | 0.00/81.0M [00:00<?, ?B/s]

Upload successful: best_model.pth (81MB)
Starting upload for file last_checkpoint.pth


100%|██████████| 81.0M/81.0M [00:00<00:00, 85.2MB/s]


Upload successful: last_checkpoint.pth (81MB)
Dataset version is being created. Please check progress at https://www.kaggle.com/datasets/dyio147/conditional-gan3d-unnormalized
  Cloud: 43/300 -> dyio147/conditional-gan3d-unnormalized
Early stopping tai epoch 43

=== TRAINING HOAN THANH ===
Best val_SSIM: 0.9805
Model luu tai: /kaggle/working/conditional_gan3d_unnormalized/best_model.pth


In [9]:
# Push da duoc tich hop vao training loop (cell tren) — chay moi epoch.
print(f'Checkpoints da duoc push len: {KAGGLE_USER}/{DATASET_NAME}')


Checkpoints da duoc push len: dyio147/conditional-gan3d-unnormalized
